In [10]:
!pip install librosa scikit-learn soundfile numpy

In [11]:
import os
import numpy as np
import librosa

from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score
from sklearn.metrics import precision_score, recall_score, f1_score


In [12]:
def extract_features(audio_path, sr=44100):

    signal, sr = librosa.load(audio_path, sr=sr)

    # =====================
    # MFCC (13)
    # =====================
    mfcc = librosa.feature.mfcc(
        y=signal,
        sr=sr,
        n_mfcc=13
    )
    mfcc_mean = np.mean(mfcc, axis=1)
    mfcc_std = np.std(mfcc, axis=1)

    # =====================
    # PLP aproximado manual
    # =====================

    # Espectrograma Mel
    mel_spec = librosa.feature.melspectrogram(
        y=signal,
        sr=sr,
        n_mels=26,
        power=2
    )

    # Compresión log
    log_mel = np.log(mel_spec + 1e-9)

    # DCT (similar a PLP)
    plp = librosa.feature.mfcc(
        S=log_mel,
        n_mfcc=13
    )

    plp_mean = np.mean(plp, axis=1)
    plp_std = np.std(plp, axis=1)

    # Concatenar
    features = np.concatenate([
        mfcc_mean, mfcc_std,
        plp_mean, plp_std
    ])

    return features


In [13]:
emotion_map = {
    "alegria": "positivo",
    "neutro": "neutral",
    "TRISTEZA": "negativo",
    "miedo": "negativo",
    "ira": "negativo",
    "asco": "negativo",
    "sorpresa": "negativo"
}

emotion2id = {
    "positivo": 0,
    "neutral": 1,
    "negativo": 2
}

def load_dataset(dataset_path):
    X, y = [], []

    for file in os.listdir(dataset_path):
        if not file.lower().endswith(".wav"):
            continue

        # Obtener emoción (segunda parte del nombre)
        emotion_raw = file.split("_")[1].lower()

        if emotion_raw not in emotion_map:
            continue

        label = emotion2id[emotion_map[emotion_raw]]

        features = extract_features(os.path.join(dataset_path, file))

        X.append(features)
        y.append(label)

    return np.array(X), np.array(y)


In [14]:
DATASET_PATH = "/content/dataset"

X, y = load_dataset(DATASET_PATH)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

model = SVC(
    kernel="rbf",
    C=10,
    gamma="scale",
    class_weight="balanced"
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='macro')
recall = recall_score(y_test, y_pred, average='macro')
f1 = f1_score(y_test, y_pred, average='macro')

print("===== MÉTRICAS GLOBALES =====")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-score : {f1:.4f}")

print(classification_report(
    y_test,
    y_pred,
    target_names=["positivo", "neutral", "negativo"]
))

# ============================
# BASELINE ALEATORIA
# ============================

print("\n===== BASELINE ALEATORIA =====")

# Generar predicciones aleatorias
random_preds = np.random.randint(0, 3, size=len(y_test))

# Métricas baseline
rand_accuracy = accuracy_score(y_test, random_preds)
rand_precision = precision_score(y_test, random_preds, average='macro', zero_division=0)
rand_recall = recall_score(y_test, random_preds, average='macro', zero_division=0)
rand_f1 = f1_score(y_test, random_preds, average='macro', zero_division=0)

print(f"Accuracy : {rand_accuracy:.4f}")
print(f"Precision: {rand_precision:.4f}")
print(f"Recall   : {rand_recall:.4f}")
print(f"F1-score : {rand_f1:.4f}")

print("\nReporte baseline:\n")
print(classification_report(
    y_test,
    random_preds,
    target_names=["positivo", "neutral", "negativo"]
))


===== MÉTRICAS GLOBALES =====
Accuracy : 0.6340
Precision: 0.6344
Recall   : 0.7070
F1-score : 0.6206
              precision    recall  f1-score   support

    positivo       0.32      0.71      0.44        51
     neutral       0.71      0.95      0.81        95
    negativo       0.87      0.47      0.61       201

    accuracy                           0.63       347
   macro avg       0.63      0.71      0.62       347
weighted avg       0.75      0.63      0.64       347


===== BASELINE ALEATORIA =====
Accuracy : 0.3718
Precision: 0.3600
Recall   : 0.3554
F1-score : 0.3362

Reporte baseline:

              precision    recall  f1-score   support

    positivo       0.15      0.35      0.21        51
     neutral       0.31      0.31      0.31        95
    negativo       0.63      0.41      0.49       201

    accuracy                           0.37       347
   macro avg       0.36      0.36      0.34       347
weighted avg       0.47      0.37      0.40       347



In [ ]:
#Carga del dataset
DATASET_PATH = "/content/dataset"

X, y = load_dataset(DATASET_PATH)

print("Muestras totales:", X.shape)

In [ ]:
#Division de train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [ ]:
#Modelo SVM
model = SVC(
    kernel="rbf",
    C=10,
    gamma="scale",
    class_weight="balanced"
)

model.fit(X_train, y_train)

In [ ]:
#Evaliacion
y_pred = model.predict(X_test)

print("\nAccuracy:", accuracy_score(y_test, y_pred))

print("\nReporte de clasificación:\n")
print(classification_report(
    y_test,
    y_pred,
    target_names=["positivo", "neutral", "negativo"]
))

# ============================
# BASELINE ALEATORIA
# ============================

print("\n===== BASELINE ALEATORIA =====")

# Generar predicciones aleatorias
random_preds = np.random.randint(0, 3, size=len(y_test))

# Métricas baseline
rand_accuracy = accuracy_score(y_test, random_preds)
rand_precision = precision_score(y_test, random_preds, average='macro', zero_division=0)
rand_recall = recall_score(y_test, random_preds, average='macro', zero_division=0)
rand_f1 = f1_score(y_test, random_preds, average='macro', zero_division=0)

print(f"Accuracy : {rand_accuracy:.4f}")
print(f"Precision: {rand_precision:.4f}")
print(f"Recall   : {rand_recall:.4f}")
print(f"F1-score : {rand_f1:.4f}")

print("\nReporte baseline:\n")
print(classification_report(
    y_test,
    random_preds,
    target_names=["positivo", "neutral", "negativo"]
))
